# Code Overlay Watcher-Maintained Generation Snapshot

**Status:** Design revised after exact external-package source review; implementation requires plan review.

**Revision epic:** `bd-hxut`  
**Supersedes:** the registered-mutator epoch lease and synchronous fsmonitor-token assumptions  
**Externally inspected revisions:** `watchman-client 0.9.0`, `notify 8.2.0`, `tributary-fs 0.1.0`, `blit-fssync 0.44.0`  
**Historical solves:** `sol_c3dcb70186b24f3a`, `sol_72bac8d1a8324f21` — retained as prior evidence, not authority for this revision

## Executive decision

Do not require agents, editors, or shell commands to register mutations. A per-root generation actor automatically maintains the overlay from filesystem change observations.

At startup or after uncertainty, the actor performs an exact Git observation, builds a complete immutable generation, and establishes a synchronized Watchman clock. It then consumes continuous subscription/`since` deltas, rebuilds only the changed dependency closure, and atomically publishes the next `Arc<OverlayGeneration>`. A warm `code_*` request performs no Git subprocess and no synchronization cookie: it acquires one trusted generation lease, pins that immutable generation, and executes entirely against it.

### Consistency contract

- Every returned fast-path response is internally consistent with one immutable overlay generation.
- Watcher-backed freshness is automatic and bounded, not a portable proof that every write completed before request start has already reached the watcher.
- A cheap end-of-request lease check detects generation publication or known trust revocation; it does not pretend to be a filesystem fence.
- Watcher disconnect, fresh-instance/recrawl, overflow/rescan, baseline loss, or build failure revokes trust and invokes exact Git recovery.
- Callers requiring strict point-in-time freshness must use the exact route; that stronger contract cannot generally guarantee sub-10-ms latency.

**Native NS-Mermaid profiles pinned:** `architecture_policy@1`, `relational_lia@1`, `state_invariant_lia@1`, and `sequence_trace@1`.

## Problem and measured evidence

The immutable overlay-generation architecture is already doing its job. A warm symbol query against a pinned generation is measured in microseconds, while end-to-end MCP latency is dominated by two exact Git observations around the query.

| Stage | Fresh measurement | Interpretation |
|---|---:|---|
| Pinned generation query | 0.0004–0.0022 ms p50 | merge/filter/sort/dedup are absent from the warm path |
| Exact Git observation A | 33.5–33.8 ms p50 | pre-query validation |
| Exact Git observation B | 33.5–33.8 ms p50 | post-query validation |
| Warm full MCP | 82.4–84.7 ms p50 | approximately 67 ms comes from exact validation |
| Warm rebuild/base-load/finalization | 0 observed | cache, generation pinning, and precomputed indexes work |

The remaining design question is therefore not query execution; it is how to remove request-time exact validation without modifying normal agent/editor/shell behavior.

### Exact external-source evaluation

| Exact package | Verified source contract | Design consequence |
|---|---|---|
| `watchman-client 0.9.0` | `Client::clock` supports a synchronized cookie; `subscribe` and query `since` provide continuous deltas; `QueryResult::is_fresh_instance` requires full state replacement. `SyncTimeout::DisableCookie` saves about 15 ms but may return a slightly outdated view. | Synchronize once at initialization/recovery, then maintain the generation asynchronously. Never pay a cookie per `code_*` request. |
| `notify 8.2.0` | Public `Watcher` exposes watch management but no clock, fence, flush, or barrier. macOS dropped-event flags become `Rescan`. | Use only as dirty/rescan hints; it cannot prove a portable request boundary. |
| `tributary-fs 0.1.0` | Implements typed sync-cookie obligations and epoch-bumped rescans, but the macOS backend writes a cookie and waits for ordinary FSEvents; it has no `FSEventStreamFlushSync` and no macOS cookie integration test. It requires Rust 1.95 while Spur uses 1.88. | Useful design evidence, not an admissible dependency or proven macOS fence. |
| `blit-fssync 0.44.0` | Arms before scan, converts watcher traffic into `Dirty`/`Rescan`, verifies against the filesystem, and publishes immutable `Arc<Index>` snapshots with `changed` and `recheck` sets. It exposes no synchronization barrier. | Adopt the shared-root reconciler and immutable publication pattern. |

The old registered-writer epoch design is rejected: it would require every modifying tool to cooperate and still would not cover human editors or a second terminal. The revised design makes observation an infrastructure responsibility and states bounded freshness honestly.

## Goals, non-goals, and terms

### Goals

- Preserve overlay correctness by returning each fast-path result from one immutable generation.
- Observe changes automatically; agents, editors, build tools, and shell commands require no mutation registration.
- Remove both exact Git observations from trusted, unchanged, warm requests.
- Maintain one shared change stream and generation actor per canonical root.
- Rebuild only changed paths and their dependency closure, retaining structural sharing.
- Fail closed to exact Git recovery on lost continuity, fresh instance/recrawl, overflow/rescan, disconnect, or build failure.
- Target warm end-to-end p95 below 10 ms with zero request-time exact observations.
- Measure and publish the bounded-freshness SLA separately from snapshot correctness.

### Non-goals

- Claiming that an internal epoch read proves the filesystem has no queued events.
- Treating `notify` or Git's public fsmonitor extension as a synchronous portable fence.
- Requiring a mutator API around normal filesystem writes.
- Removing the exact Git implementation.
- Using TTL or mtime as a correctness authority.
- Promising strict point-in-time freshness on the watcher route.

### Terms

- **Generation actor:** the sole per-root owner of watcher continuity, changed paths, rebuild scheduling, trust state, and atomic publication.
- **Generation lease:** an immutable `Arc<OverlayGeneration>` plus the actor trust token observed at acquisition.
- **Trusted generation:** a published generation descended from a successful exact baseline through an unbroken observed delta sequence.
- **Bounded freshness:** the configured and measured maximum acceptable observation/rebuild lag; distinct from snapshot consistency.
- **Exact baseline:** a complete Git observation and overlay build used at startup or recovery.
- **Continuity loss:** any fresh instance/recrawl, overflow/rescan, disconnect, backend error, or unknown provider state that prevents incremental proof.
- **Strict route:** the existing exact Git validation path for callers or states that require point-in-time freshness.

## Validation-route eligibility

The generation-snapshot route is available only when four runtime predicates agree:

1. A change provider is running for the canonical root.
2. A successful exact baseline has established the current root/Git identity.
3. Provider continuity has remained valid since that baseline.
4. The actor has atomically published a trusted immutable generation.

If any predicate is false or unknown, the route is `exact_fallback`. Configuration may request `Auto`, but it cannot manufacture trust.

Provider policy is explicit:

- **Watchman:** preferred provider. Establish one synchronized clock during initialization/recovery, then consume subscription/`since` deltas without per-request cookies.
- **Notify:** fallback hint provider. Any error or `need_rescan` revokes trust and starts exact recovery. Its route is bounded-freshness snapshot consistency, never advertised as a hard filesystem fence.
- **No provider or unsupported filesystem:** exact route.

Shared roots and multiple human/agent processes do not require special writer registration. They are safe under the same snapshot contract because all writes are observed beneath the root; their higher change rate may reduce cache reuse but does not alter routing semantics.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-GENERATION-ELIGIBILITY
@type ValidationRoute = enum[generation_snapshot, exact_fallback]
@input watcher_ready: Bool
@input source_set_complete: Bool
@input exact_baseline_valid: Bool
@input continuity_valid: Bool
@input generation_trusted: Bool
@output status: ValidationRoute
@requires PRE: true`"]

    SNAPSHOT["`@branch SNAPSHOT
@when watcher_ready and source_set_complete and exact_baseline_valid and continuity_valid and generation_trusted
@ensures SNAPSHOT_ROUTE: status = generation_snapshot`"]

    EXACT["`@branch EXACT
@when not (watcher_ready and source_set_complete and exact_baseline_valid and continuity_valid and generation_trusted)
@ensures EXACT_ROUTE: status = exact_fallback`"]

    CHECK["`@verify ELIGIBILITY_DETERMINISTIC: prove determinism
@verify ELIGIBILITY_COVERAGE: prove partition_coverage
@verify ELIGIBILITY_EXCLUSIVE: prove partition_exclusive
@verify ELIGIBILITY_STATUSES: witness each status
@verify SNAPSHOT_REACHABLE: witness branch SNAPSHOT
@verify EXACT_REACHABLE: witness branch EXACT`"]

    SPEC --> SNAPSHOT --> CHECK
    SPEC --> EXACT --> CHECK

## Target architecture and ownership

The verified architecture cell below is the authoritative topology. It separates change sources, the generation control plane, and the immutable query plane.

### Component responsibilities

- `ChangeProvider` is a pluggable source-set observer. It watches the canonical worktree plus every resolved Git administrative path (`gitdir`, `commondir`, index, HEAD, refs, excludes, sparse-checkout, and submodule metadata). `WatchmanProvider` is primary; `NotifyProvider` supplies conservative dirty/rescan hints when Watchman is unavailable.
- `ExactGitReconciler` establishes the startup/recovery baseline and is the only component allowed to restore trust after continuity loss.
- `RootGenerationActor` serializes delta application, trust revocation, rebuild scheduling, and atomic state publication. It is infrastructure-owned, not an agent mutator. Events that arrive during a rebuild advance the observed target and cancel, extend, or supersede the obsolete build before publication.
- `ChangedPathSet` coalesces normalized create, modify, delete, rename, Git metadata, ignore, submodule, and sparse-checkout changes.
- `IncrementalGenerationBuilder` reuses the prior compatible generation, rebuilding affected chunks, visible indexes, and dependency-closure adjacency.
- `OverlayGeneration` remains immutable and structurally shared. `PublishedState { generation, token, trust, source_clock }` is one immutable value behind one atomically swapped `Arc`; generation and trust are never published through independent atomics.
- `CodeDispatcher` acquires one generation lease. `PinnedGenerationClient` performs the query without merging, sorting, filtering, deduplicating, invoking Git, or synchronizing the watcher.
- `ResponseGate` rechecks the identity/trust of the same atomic `PublishedState`. A changed state may retry once; revoked trust invokes exact fallback. This check detects internal publication, not unobserved filesystem traffic.

### Provider decision

`watchman-client 0.9.0` is the recommended primary integration because it provides clocks, subscriptions, incremental `since`, and fresh-instance recovery. `notify 8.2.0` remains a portability fallback. `tributary-fs 0.1.0` and `blit-fssync 0.44.0` are architecture references only.

In [ ]:
architecture-beta
    group public(cloud)[Observed sources and clients]
    group edge(cloud)[Generation control plane]
    group private(cloud)[Immutable query plane]

    service client(internet)[MCP client] in public
    service worktree(disk)[Canonical worktree] in public
    service gitdir(disk)[Resolved worktree gitdir] in public
    service commondir(disk)[Resolved shared commondir] in public

    service watchman(server)[Watchman subscriptions] in edge
    service notify(server)[Notify fallback] in edge
    service changefeed(queue)[Normalized change feed] in edge
    service exact(server)[Exact Git reconciler] in edge
    service actor(server)[Root generation actor] in edge
    service changedset(disk)[Changed path set] in edge
    service builder(server)[Incremental builder] in edge
    service dispatcher(server)[code_* dispatcher] in edge

    service published(database)[Atomic Arc PublishedState] in private
    service query(server)[Pinned generation query] in private
    service gate(server)[Response gate] in private

    client:R --> L:dispatcher
    worktree:R --> L:watchman
    worktree:R --> L:notify
    gitdir:R --> L:watchman
    gitdir:R --> L:notify
    commondir:R --> L:watchman
    commondir:R --> L:notify
    worktree:R --> L:exact
    gitdir:R --> L:exact
    commondir:R --> L:exact
    watchman:R --> L:changefeed
    notify:R --> L:changefeed
    changefeed:R --> L:actor
    exact:R --> L:actor
    actor:R --> L:changedset
    changedset:R --> L:builder
    builder:R --> L:published
    dispatcher:R --> L:actor
    dispatcher:R --> L:query
    dispatcher:R --> L:gate
    published:R --> L:query
    published:R --> L:gate
    query:R --> L:gate
    dispatcher:R --> L:exact
    exact:R --> L:gate

In [ ]:
stateDiagram-v2
    [*] --> Untrusted
    Untrusted --> Trusted: recover_exact
    Trusted --> Rebuilding: change_batch
    Rebuilding --> Rebuilding: change_batch
    Rebuilding --> Trusted: publish_generation
    Trusted --> Untrusted: invalidate
    Rebuilding --> Untrusted: invalidate

    note right of Untrusted
      @spec CODE-OVERLAY-GENERATION-LIFECYCLE
      @type GenerationEvent = enum[recover_exact, change_batch, publish_generation, invalidate]
      @input event: GenerationEvent
      @state-var observed_revision: Int
      @state-var build_revision: Int
      @state-var published_revision: Int
      @state-var trusted: Bool
      @requires PRE: observed_revision = 0 and build_revision = 0 and published_revision = 0 and trusted = false
      @state Untrusted
      @invariant ORDERED: observed_revision >= 0 and build_revision >= 0 and published_revision >= 0 and published_revision <= build_revision and build_revision <= observed_revision
      @invariant TRUSTED_MATCH: not trusted or (published_revision = observed_revision and build_revision = observed_revision)
      @verify ORDERED_INIT: prove initiate ORDERED
      @verify TRUSTED_INIT: prove initiate TRUSTED_MATCH
      @verify ORDERED_RECOVER: prove preserve ORDERED on RECOVER_EXACT
      @verify ORDERED_CHANGE: prove preserve ORDERED on CHANGE_BATCH
      @verify ORDERED_QUEUE: prove preserve ORDERED on QUEUE_BATCH
      @verify ORDERED_PUBLISH: prove preserve ORDERED on PUBLISH_GENERATION
      @verify ORDERED_INVALIDATE_TRUSTED: prove preserve ORDERED on INVALIDATE_TRUSTED
      @verify ORDERED_INVALIDATE_REBUILDING: prove preserve ORDERED on INVALIDATE_REBUILDING
      @verify TRUSTED_RECOVER: prove preserve TRUSTED_MATCH on RECOVER_EXACT
      @verify TRUSTED_CHANGE: prove preserve TRUSTED_MATCH on CHANGE_BATCH
      @verify TRUSTED_QUEUE: prove preserve TRUSTED_MATCH on QUEUE_BATCH
      @verify TRUSTED_PUBLISH: prove preserve TRUSTED_MATCH on PUBLISH_GENERATION
      @verify TRUSTED_INVALIDATE_TRUSTED: prove preserve TRUSTED_MATCH on INVALIDATE_TRUSTED
      @verify TRUSTED_INVALIDATE_REBUILDING: prove preserve TRUSTED_MATCH on INVALIDATE_REBUILDING
    end note

    note left of Trusted
      @state Trusted
      @transition RECOVER_EXACT
      @from Untrusted
      @to Trusted
      @event event = recover_exact
      @guard trusted = false and published_revision <= build_revision and build_revision <= observed_revision
      @update observed_revision' = observed_revision + 1
      @update build_revision' = observed_revision + 1
      @update published_revision' = observed_revision + 1
      @update trusted' = true
    end note

    note right of Rebuilding
      @state Rebuilding
      @transition CHANGE_BATCH
      @from Trusted
      @to Rebuilding
      @event event = change_batch
      @guard trusted = true and published_revision = observed_revision and build_revision = observed_revision
      @update observed_revision' = observed_revision + 1
      @update build_revision' = observed_revision + 1
      @update published_revision' = published_revision
      @update trusted' = false
    end note

    note left of Rebuilding
      @transition QUEUE_BATCH
      @from Rebuilding
      @to Rebuilding
      @event event = change_batch
      @guard trusted = false and published_revision <= build_revision and build_revision <= observed_revision
      @update observed_revision' = observed_revision + 1
      @update build_revision' = observed_revision + 1
      @update published_revision' = published_revision
      @update trusted' = false
    end note

    note right of Trusted
      @transition PUBLISH_GENERATION
      @from Rebuilding
      @to Trusted
      @event event = publish_generation
      @guard trusted = false and published_revision < observed_revision and build_revision = observed_revision
      @update observed_revision' = observed_revision
      @update build_revision' = build_revision
      @update published_revision' = observed_revision
      @update trusted' = true
    end note

    note left of Untrusted
      @transition INVALIDATE_TRUSTED
      @from Trusted
      @to Untrusted
      @event event = invalidate
      @guard trusted = true and published_revision = observed_revision and build_revision = observed_revision
      @update observed_revision' = observed_revision + 1
      @update build_revision' = build_revision
      @update published_revision' = published_revision
      @update trusted' = false
    end note

    note right of Untrusted
      @transition INVALIDATE_REBUILDING
      @from Rebuilding
      @to Untrusted
      @event event = invalidate
      @guard trusted = false and published_revision <= build_revision and build_revision <= observed_revision
      @update observed_revision' = observed_revision + 1
      @update build_revision' = build_revision
      @update published_revision' = published_revision
      @update trusted' = false
    end note

## Healthy-path protocol

1. Resolve the canonical worktree, `gitdir`, and `commondir`; arm subscriptions for the complete source set and capture a pre-baseline clock `c0` before enumeration.
2. Perform an exact Git observation and build the candidate immutable baseline while events continue accumulating.
3. Replay/drain every delta `since c0`, fold those changes into the candidate, and advance to a synchronized post-replay clock. Publish trust only after the exact baseline plus replay handoff is coherent; a fresh instance, recrawl, overflow, or uncovered source aborts publication.
4. Coalesce each observed delta batch into normalized changed paths. Git metadata, ignore, submodule, sparse-checkout, create/delete, and both rename sides participate.
5. Build the next generation from the prior compatible generation and changed dependency closure. The published generation remains readable while rebuilding; deltas received during the build advance its target and prevent obsolete publication.
6. Atomically swap one `Arc<PublishedState { generation, token, trust, source_clock }>` after the build target equals the latest observed revision, then clear only the changes covered by that published revision.
7. A request acquire-loads one trusted `Arc<PublishedState>` and pins that exact value for the full handler execution.
8. The response gate compares the same atomic state identity. If a newer trusted state was published during the query, it may retry once as a freshness preference. The pinned result was already snapshot-consistent.
9. A lost-continuity state never incrementally repairs trust. It performs exact reconciliation, re-arms the complete source set, replays from a pre-scan boundary, and publishes a replacement baseline.

The warm unchanged request performs two atomic/control-plane reads plus one immutable query. No Git subprocess, watcher cookie, repository walk, merge, sort, filter, or dedup occurs on that path.

In [ ]:
sequenceDiagram
    participant Git
    participant Watcher
    participant Actor
    participant Builder
    participant Dispatcher
    participant Query

    Note over Git,Query: @spec CODE-OVERLAY-GENERATION-REQUEST-PROTOCOL<br/>@input source_set_complete: Bool<br/>@input c0: Int<br/>@input baseline_revision: Int<br/>@input replay_revision: Int<br/>@input published_revision: Int<br/>@requires NONNEGATIVE: c0 >= 0 and baseline_revision >= 0 and replay_revision >= 0 and published_revision >= 0

    Actor->>Watcher: arm complete source set
    Note over Actor,Watcher: @message ARM_SOURCES<br/>@from Actor<br/>@to Watcher<br/>@event arm complete source set<br/>@order 1<br/>@when source_set_complete<br/>@ensures SOURCES_ARMED: source_set_complete

    Watcher-->>Actor: return pre-baseline clock c0
    Note over Watcher,Actor: @message CAPTURE_C0<br/>@from Watcher<br/>@to Actor<br/>@event return pre-baseline clock c0<br/>@order 2<br/>@when source_set_complete and c0 >= 0<br/>@ensures CLOCK_READY: c0 >= 0

    Git->>Actor: establish exact candidate baseline
    Note over Git,Actor: @message ESTABLISH_BASELINE<br/>@from Git<br/>@to Actor<br/>@event establish exact candidate baseline<br/>@order 3<br/>@when source_set_complete and baseline_revision >= c0<br/>@ensures BASELINE_AFTER_ARM: baseline_revision >= c0

    Watcher->>Actor: replay and drain since c0
    Note over Watcher,Actor: @message REPLAY_HANDOFF<br/>@from Watcher<br/>@to Actor<br/>@event replay and drain since c0<br/>@order 4<br/>@when replay_revision >= baseline_revision<br/>@ensures REPLAY_COVERS_BASELINE: replay_revision >= baseline_revision

    Actor->>Builder: rebuild through replay revision
    Note over Actor,Builder: @message REBUILD_THROUGH_REPLAY<br/>@from Actor<br/>@to Builder<br/>@event rebuild through replay revision<br/>@order 5<br/>@when replay_revision >= baseline_revision<br/>@ensures BUILD_TARGET_CURRENT: replay_revision >= baseline_revision

    Builder->>Actor: atomically publish Arc PublishedState
    Note over Builder,Actor: @message PUBLISH_STATE<br/>@from Builder<br/>@to Actor<br/>@event atomically publish Arc PublishedState<br/>@order 6<br/>@when published_revision = replay_revision<br/>@ensures PUBLISHED_AFTER_REPLAY: published_revision = replay_revision

    Dispatcher->>Actor: acquire trusted PublishedState
    Note over Dispatcher,Actor: @message ACQUIRE_STATE<br/>@from Dispatcher<br/>@to Actor<br/>@event acquire trusted PublishedState<br/>@order 7<br/>@when published_revision = replay_revision<br/>@ensures LEASE_CURRENT: published_revision = replay_revision

    Actor->>Query: pin immutable PublishedState
    Note over Actor,Query: @message PIN_STATE<br/>@from Actor<br/>@to Query<br/>@event pin immutable PublishedState<br/>@order 8<br/>@when published_revision = replay_revision<br/>@ensures PINNED: published_revision = replay_revision

    Dispatcher->>Query: execute code query
    Note over Dispatcher,Query: @message CODE_QUERY<br/>@from Dispatcher<br/>@to Query<br/>@event execute code query<br/>@order 9<br/>@when published_revision = replay_revision<br/>@ensures QUERY_SNAPSHOT: true

    Query-->>Dispatcher: return snapshot-consistent result
    Note over Query,Dispatcher: @message RETURN_RESULT<br/>@from Query<br/>@to Dispatcher<br/>@event return snapshot-consistent result<br/>@order 10<br/>@when published_revision = replay_revision<br/>@ensures RESPONSE_SNAPSHOT: published_revision = replay_revision

    Note over Git,Query: @verify GENERATION_REQUEST_TRACE: prove sequence_protocol

## Response commit gate

The response gate protects the declared snapshot contract; it does not claim a portable filesystem fence.

| Condition | Action |
|---|---|
| Snapshot route, generation trusted, lease unchanged | Commit pinned snapshot |
| Snapshot route, newer trusted generation published, first attempt | Discard and retry once as a freshness preference |
| Snapshot route, repeated lease change | Exact fallback |
| Trust revoked at any point | Exact fallback |
| Exact route selected before query | Exact Git validation |
| Exact validation succeeds | Commit exact result |
| Exact validation fails | Return an error; never return the speculative snapshot |

A pinned generation remains internally correct even if another generation is published during the query. The optional retry reduces visible staleness; it is not necessary for structural consistency.

The graph may be stale while the overlay is newer: the overlay remains the correction layer and source of the visible code result. What changes in this design is the freshness claim. The watcher route returns the latest trusted generation known to the actor, subject to the measured bounded-freshness SLA. Strict “all writes completed before request start” semantics remain exact-route only.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-GENERATION-RESPONSE-GATE
@type CommitAction = enum[commit_snapshot, retry, commit_exact, error]
@input published_state_trusted: Bool
@input same_published_state: Bool
@input first_attempt: Bool
@input exact_route: Bool
@input exact_ok: Bool
@output status: CommitAction
@requires PRE: true`"]

    COMMIT["`@branch COMMIT
@when not exact_route and published_state_trusted and same_published_state
@ensures COMMIT_STATUS: status = commit_snapshot`"]

    RETRY["`@branch RETRY
@when not exact_route and published_state_trusted and not same_published_state and first_attempt
@ensures RETRY_STATUS: status = retry`"]

    EXACT["`@branch EXACT
@when (exact_route or not published_state_trusted or (not exact_route and published_state_trusted and not same_published_state and not first_attempt)) and exact_ok
@ensures EXACT_STATUS: status = commit_exact`"]

    ERROR["`@branch ERROR
@when (exact_route or not published_state_trusted or (not exact_route and published_state_trusted and not same_published_state and not first_attempt)) and not exact_ok
@ensures ERROR_STATUS: status = error`"]

    CHECK["`@verify GATE_DETERMINISTIC: prove determinism
@verify GATE_COVERAGE: prove partition_coverage
@verify GATE_EXCLUSIVE: prove partition_exclusive
@verify GATE_STATUSES: witness each status
@verify COMMIT_REACHABLE: witness branch COMMIT
@verify RETRY_REACHABLE: witness branch RETRY
@verify EXACT_REACHABLE: witness branch EXACT
@verify ERROR_REACHABLE: witness branch ERROR`"]

    SPEC --> COMMIT --> CHECK
    SPEC --> RETRY --> CHECK
    SPEC --> EXACT --> CHECK
    SPEC --> ERROR --> CHECK

## Cache, invalidation, and concurrency semantics

- **No TTL correctness rule.** Immutable generations remain addressable until capacity eviction. Freshness comes from provider continuity and publication, not age.
- **One actor per canonical root.** Provider events, trust transitions, coalescing, rebuilds, and publication are serialized without intercepting writers.
- **Arm, clock, scan, replay.** Subscribe to the canonical worktree and resolved `gitdir`/`commondir` source set, capture `c0`, perform the exact scan, then replay/drain `since c0` before publishing trust. This closes the initialization gap.
- **Existing cache preserved.** Identity-keyed lookup, compatible seeding, singleflight construction, persistent generations, and atomic publication continue to own reuse.
- **Changed-set rebuild.** Dirty paths drive affected file chunks, symbols, visibility indexes, and dependency-closure adjacency. Unchanged segments retain `Arc` identity. Deltas received during a build advance its target and prevent an obsolete target from publishing.
- **Immutable request pin.** Queries hold an `Arc`; they never hold the actor or rebuild lock.
- **Atomic publication.** Publish one immutable `Arc<PublishedState { generation, token, trust, source_clock }>` with one atomic swap. Requests acquire-load that one value; no split Arc/token read is permitted.
- **Known-loss revocation.** Fresh instance/recrawl, overflow/rescan, disconnect, provider error, baseline mismatch, or failed rebuild makes the actor untrusted before recovery.
- **Racily clean data.** Recent/ambiguous metadata must be re-read or hashed before identity is adopted; borrow the explicit recheck-set pattern from `blit-fssync`.
- **Git-wide invalidation.** Resolve and observe the worktree root, linked-worktree `gitdir`, shared `commondir`, index, HEAD/refs, ignore/exclude rules, sparse checkout, and submodule metadata. If any required administrative source cannot be observed continuously, revoke trust and reconcile exactly.
- **Bounded queues.** Event coalescing may collapse duplicate dirty hints, but a loss/overflow signal dominates the batch and cannot be dropped.
- **Epoch limitation.** Two equal actor epochs prove only that no generation was published or trust revoked between reads. They do not prove the OS event queue was empty.

## Failure and fallback matrix

| Situation | Required behavior |
|---|---|
| Startup with provider available | Resolve worktree + Git administrative sources; arm and capture `c0`; exact baseline; replay/drain `since c0`; then publish trusted state |
| Trusted unchanged generation | Pin and query with zero request-time exact observations |
| Delta arrives during query | Existing pin remains snapshot-consistent; optionally retry once on changed state |
| Delta arrives during rebuild | Advance/supersede the build target; never publish an obsolete revision |
| Rapid edits from agents, editors, builds, or second terminals | Coalesce changed paths and publish generations; no writer registration |
| Watchman incremental delta | Rebuild changed closure and advance stored clock |
| Watchman fresh instance/recrawl or connection loss | Revoke trust, discard incremental continuity, exact rebuild |
| Notify `need_rescan` or any watcher error | Revoke trust and exact rebuild |
| Provider queue overflow or coalescing with loss | Loss dominates dirty hints; exact rebuild |
| No supported provider | Exact route |
| Linked-worktree `gitdir`/`commondir`, HEAD/index/ignore/submodule/sparse-checkout change | Observe as part of the resolved source set; otherwise revoke trust and reconcile; never mtime-only |
| Racily clean or unstable file read | Recheck/hash before publishing identity |
| Generation build fails or continuity is lost while rebuilding | Keep prior generation addressable but revoke trust for new requests; transition to exact recovery or error |
| Exact observation fails | Error; do not return an untrusted speculative result |
| Strict point-in-time caller | Exact route even when watcher is healthy |

Normal multi-process activity is no longer classified as unsafe merely because ownership is shared. It increases the event rate and therefore the probability of a freshness retry, while snapshot safety continues to come from immutable pinning.

## Release gate

The optimization stays probe-only until correctness, recovery, freshness, and performance pass together.

- Provider contract tests cover Watchman clock/subscription/`since`, fresh-instance replacement, disconnect, and exact recovery.
- Notify tests prove every error and `need_rescan` becomes trust revocation and full reconciliation; no test calls it a hard fence.
- Initialization tests arm observation before the exact scan and stress writes across the scan/clock/publish handoff.
- Snapshot tests cover a publish during query, trust revocation during query, retry, and exact failure.
- Cross-project exact-oracle comparison reports zero result mismatches.
- Platform stress tests include macOS event coalescing/loss and Linux overflow; no undocumented backend ordering is assumed.
- A measured bounded-freshness SLA is reported independently from query latency.
- A 30-run small/medium/large matrix reports warm p95 below 10 ms and zero request-time exact Git observations.
- Exact fallback and recovery latency remain measured and available.

The sub-10-ms threshold applies to trusted warm snapshot requests. It is not a latency promise for initialization, recovery, strict exact requests, or a provider synchronization cookie.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-GENERATION-RELEASE
@type ReleaseDecision = enum[release, probe_only]
@input provider_contract_tests_pass: Bool
@input startup_handoff_tests_pass: Bool
@input git_metadata_watch_tests_pass: Bool
@input atomic_publication_tests_pass: Bool
@input recovery_tests_pass: Bool
@input fallback_tests_pass: Bool
@input snapshot_tests_pass: Bool
@input platform_stress_tests_pass: Bool
@input bounded_staleness_sla_pass: Bool
@input correctness_mismatches: Int
@input warm_p95_us: Int
@input request_exact_observations: Int
@output status: ReleaseDecision
@requires NONNEGATIVE_METRICS: correctness_mismatches >= 0 and warm_p95_us >= 0 and request_exact_observations >= 0`"]

    RELEASE["`@branch RELEASE
@when provider_contract_tests_pass and startup_handoff_tests_pass and git_metadata_watch_tests_pass and atomic_publication_tests_pass and recovery_tests_pass and fallback_tests_pass and snapshot_tests_pass and platform_stress_tests_pass and bounded_staleness_sla_pass and correctness_mismatches = 0 and warm_p95_us < 10000 and request_exact_observations = 0
@ensures RELEASE_STATUS: status = release`"]

    PROBE["`@branch PROBE
@when not (provider_contract_tests_pass and startup_handoff_tests_pass and git_metadata_watch_tests_pass and atomic_publication_tests_pass and recovery_tests_pass and fallback_tests_pass and snapshot_tests_pass and platform_stress_tests_pass and bounded_staleness_sla_pass and correctness_mismatches = 0 and warm_p95_us < 10000 and request_exact_observations = 0)
@ensures PROBE_STATUS: status = probe_only`"]

    CHECK["`@verify RELEASE_DETERMINISTIC: prove determinism
@verify RELEASE_COVERAGE: prove partition_coverage
@verify RELEASE_EXCLUSIVE: prove partition_exclusive
@verify RELEASE_STATUSES: witness each status`"]

    SPEC --> RELEASE --> CHECK
    SPEC --> PROBE --> CHECK

## TDD and PRE → POST evaluation contract

Implementation must use strict RED → GREEN and record solver-backed PRE and POST evaluation for each task.

### Required RED contracts

- The provider is armed before baseline enumeration; writes across initialization cannot disappear.
- A synchronized Watchman clock is paid at initialization/recovery, never per `code_*` request.
- Subscription/`since` clocks advance monotonically; `is_fresh_instance` discards prior incremental state.
- Notify error/`need_rescan` and simulated queue overflow revoke trust before another fast-path acquisition.
- No agent, editor, or shell mutation requires a mutator registration call.
- Changed paths rebuild only affected dependency closure; unchanged segments preserve `Arc` identity.
- A query pins one immutable generation and remains structurally consistent across concurrent publication.
- A changed trusted lease retries at most once; revoked trust invokes exact fallback.
- Exact failure never returns the speculative snapshot.
- Racily clean, rename, create/delete, HEAD/index/ignore/submodule/sparse-checkout cases match the exact oracle.

### Performance and freshness evaluation

Capture PRE and POST stage timings for at least 30 runs on three project sizes: lease acquisition, generation lookup/build, pinned query, lease recheck, retry/fallback, serialization, and total MCP latency. Report p50/p95, request-time exact-observation count, generation-build count, recovery count, provider lag from write to trusted publication, lease-change count, and oracle mismatch count.

Run write-to-query stress distributions separately from warm-query latency. A fast stale result is not a passing freshness result, and a proof does not replace measurement.

## Migration and rollout

1. **Instrument exact baseline:** add root/provider state, generation epoch, trust, fresh-instance/rescan, rebuild, provider-lag, route, retry, and fallback diagnostics while exact A/B remains authoritative.
2. **Introduce the root actor:** arm observation before baseline scan, publish immutable generations, but continue returning exact results.
3. **Probe snapshot route:** execute the watcher-maintained generation query and compare with exact validation without affecting returned results.
4. **Release Watchman route:** allow trusted snapshot commits only after provider, recovery, platform-stress, freshness-SLA, oracle, and latency gates pass.
5. **Release notify fallback separately:** label it bounded-freshness; require conservative rescan/error behavior and its own platform matrix.
6. **Retain strict fallback:** keep exact validation for recovery, unsupported providers, debugging, rollback, and callers requesting point-in-time freshness.

Rollback is configuration-only: disable snapshot commit and route every request through exact validation. Historical immutable generations need not be deleted.

A periodic exact audit may be added as defense in depth, but its cadence must come from measured provider-loss/freshness data and a bounded-risk objective. TTL is never the correctness mechanism.

## Implementation boundaries and dependency DAG

This specification intentionally stops before file-level task assignment. The implementation plan must re-ground symbols against the then-current graph.

1. **Provider abstraction and root actor** — canonical root registry, provider lifecycle, trust state, generation epoch, bounded/coalescing ingress, diagnostics.
2. **Exact baseline and recovery** — arm-before-scan handoff, exact Git observation, Watchman synchronized clock, fresh-instance/rescan/disconnect recovery. Depends on 1.
3. **Continuous delta and changed set** — Watchman subscription/`since`, notify Dirty/Rescan mapping, Git metadata coverage, normalized changed paths. Depends on 1 and 2.
4. **Incremental generation publication** — compatible seed, changed dependency closure, racy recheck set, singleflight build, atomic `Arc` publish. Depends on 3.
5. **MCP generation lease** — trusted acquisition, pinned request client, optional one retry on newer trusted publication, exact fallback on revoked trust; preserve existing request cache. Depends on 2 and 4.
6. **Oracle, platform, freshness, and performance gates** — initialization races, loss recovery, cross-project correctness, write-to-publication lag, 30-run latency matrix, release decision. Depends on 5.

Expected scope is primarily `spur-graph`, with a small provider/configuration seam where process startup already owns graph services. The plan must not move query-index logic out of `spur-graph` or add wrappers around agent/editor filesystem operations.

## Formal proof and source evidence

All six native contracts were executed after the final target revision. Their stored sources, metadata, and proof inputs are fresh; all **78 / 78** generated obligations matched their expected solver statuses.

| Contract | Profile | Obligations | Result | Source hash | Report hash |
|---|---|---:|---|---|---|
| `CODE-OVERLAY-GENERATION-ELIGIBILITY` | `relational_lia@1` | 7 / 7 | verified | `ea742f53c8ebf2465f864711e55136c4a6e75b0b8d34029c214520d32b0d4034` | `962892bbd35966bdf3b79cf87b17aa2a988d84698abca989eb44834514c74962` |
| target architecture topology | `architecture_policy@1` | 26 / 26 | verified | `30beed64423380c68122fdf0d81fd9d802e6d58331d09fc4722603443da76e40` | `b9ee807e677ef1c9c958bd7153f9e84796d5f5e59dc1955dde842d5c150baab6` |
| `CODE-OVERLAY-GENERATION-LIFECYCLE` | `state_invariant_lia@1` | 28 / 28 | verified | `8a96d0467c8eae7bac7574bb81cfe72d8e32736696aec91e04d25ed566c4e05e` | `1650cd1a1c081f175e153e6b27dd4ef16fcb0b8d8fd2f8f4bacf1d63a995a40f` |
| `CODE-OVERLAY-GENERATION-REQUEST-PROTOCOL` | `sequence_trace@1` | 1 / 1 | verified | `fac9ba5b5629d041756e247a5ac6de20fd7e6dd7683fe1c89d52cfdae66f30bd` | `ac24b1239a89e705342639f4f1b1631b01b2fc44d633fe58bc31f14882554ec0` |
| `CODE-OVERLAY-GENERATION-RESPONSE-GATE` | `relational_lia@1` | 11 / 11 | verified | `9df59c1c51a83d8d5a48de2b171aa2d0d24e10ea2304a136453537805f4727aa` | `74e83a494c354b1aea21eaa491c5e0d96356557895812a5915f6dd9e6521fbe7` |
| `CODE-OVERLAY-GENERATION-RELEASE` | `relational_lia@1` | 5 / 5 | verified | `55499878279c65c840cfaf91d560e408fd7ebf794e7cfefdcdc2ce08abd6b38e` | `19a9112015aedb64648902b0366244938150e0bc3e986accff4b8bd84cf1edd9` |

### Exact external-source grounding

- `watchman-client 0.9.0`: primary provider; source-verified clocks, subscriptions, incremental `since`, and mandatory fresh-instance recovery. A synchronized clock belongs to initialization/recovery, not each request.
- `notify 8.2.0`: portability fallback; source-verified dirty/rescan hints but no fence, barrier, flush, or clock. It therefore cannot independently establish an exact handoff.
- `tributary-fs 0.1.0`: useful typed cookie/loss/rescan reference, but its macOS path is ordinary FSEvents observation and its MSRV exceeds Spur's current toolchain.
- `blit-fssync 0.44.0`: useful architecture reference for arm-before-scan, serialized root ownership, dirty/rescan recovery, filesystem verification, changed/recheck sets, and immutable `Arc<Index>` publication. It does not supply a request-time barrier.

### Proof boundary and superseded evidence

These proofs establish internal branch partitions, state invariants, ordered handoff messages, zoning/non-vacuity, and release-gate completeness. They do **not** prove platform event latency or watcher completeness; those remain mandatory TDD, stress, and bounded-staleness release measurements.

Historical solves `sol_c3dcb70186b24f3a` and `sol_72bac8d1a8324f21` are retained only as records of the discarded registered-writer epoch design. They no longer authorize the target. The recommended architecture requires no agent/editor/shell mutation registration and derives trust only from the automatic source-set observer plus exact recovery.